In [ ]:
import importlib
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import python.benchmark_reporting.constants as constants
import python.benchmark_reporting.io as reporting_io
import python.benchmark_reporting.transforms as transforms
import python.benchmark_reporting.plotting as plotting

importlib.reload(constants)
importlib.reload(reporting_io)
importlib.reload(transforms)
importlib.reload(plotting)

from python.benchmark_reporting.constants import *
from python.benchmark_reporting.io import (
    load_benchmark_data,
    load_speedup_summary,
    load_lloyd_parity_summary,
    load_gmm_parity_summary,
)
from python.benchmark_reporting.transforms import *
from python.benchmark_reporting.plotting import *

DATA_DIR = Path("datasets")

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.autolayout": True, "figure.dpi": 120})

pd.set_option('display.max_columns', 50)

In [ ]:
df_bench = load_benchmark_data(
    DATA_DIR,
    time_field="time_s",
    statistic="median",
)

df_speedup = load_speedup_summary(
    DATA_DIR,
    time_field="time_per_algorithm_iteration_s",
    ratio_statistic="median_ratio",
)

print(f"Loaded {len(df_bench)} summarized timing rows.")
print(f"Loaded {len(df_speedup)} post-processed speedup rows.")

display(df_bench.head())
display(df_speedup.head())

In [ ]:
lloyd_tolerance_pct = 1e-6

df_lloyd_parity = load_lloyd_parity_summary(
    data_dir=DATA_DIR,
)

df_gmm_parity = load_gmm_parity_summary(
    data_dir=DATA_DIR,
)

if df_lloyd_parity.empty:
    print("WARNING: No Lloyd parity records found in benchmark_summary.json.")
else:
    lloyd_failures = df_lloyd_parity[df_lloyd_parity["Status"] == "❌ FAIL"]

    if not lloyd_failures.empty:
        print(
            f"WARNING: Found {len(lloyd_failures)} Lloyd configurations outside "
            f"{lloyd_tolerance_pct}% inertia-diff limit!"
        )
        display(lloyd_failures)
    else:
        print(
            f"SUCCESS: All {len(df_lloyd_parity)} Lloyd configurations match "
            f"within {lloyd_tolerance_pct}% inertia-diff limit!"
        )

if df_gmm_parity.empty:
    print("WARNING: No GMM parity records found in benchmark_summary.json.")
else:
    gmm_failures = df_gmm_parity[df_gmm_parity["Status"] == "❌ FAIL"]

    if not gmm_failures.empty:
        print(f"WARNING: Found {len(gmm_failures)} GMM parity failures!")
        display(gmm_failures)
    else:
        print(f"SUCCESS: All {len(df_gmm_parity)} GMM configurations pass parity!")

In [ ]:
df_bench = add_time_per_algorithm_iteration_columns(df_bench)


def format_variant_params_title(variant, params):
    if pd.isna(params) or str(params) == "Default":
        return f"Variant: {variant}"
    return f"Variant: {variant}; Params: {params}"


df_lloyd = filter_bench(
    df_bench,
    phase=PHASE_MAP["lloyd"],
)

df_gmm = filter_bench(
    df_bench,
    phase=PHASE_MAP["gmm"],
)

df_speedup = prepare_speedup_comparison_data(df_speedup)


In [ ]:
from python.benchmark_reporting.plotting import *
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter


fixed_phases = [
    PHASE_MAP["soa"],
    PHASE_MAP["pp"],
]

df_cpp = filter_bench(
    df_bench,
    language=LANG_CPP,
).copy()

lloyd_N_values = set(
    filter_bench(
        df_cpp,
        phase=PHASE_MAP["lloyd"],
    )[COL_SAMPLES].dropna().unique()
)

fixed_N_values = set(
    filter_bench(
        df_cpp,
        phase=fixed_phases,
    )[COL_SAMPLES].dropna().unique()
)

eligible_N_values = sorted(lloyd_N_values & fixed_N_values)

if not eligible_N_values:
    print(
        "Skipping fixed-cost vs Lloyd algorithm-iteration graph: "
        "requires C++ Lloyd plus at least one of C++ SoA / C++ K-means++."
    )
else:
    max_N = max(eligible_N_values)

    df_math_summary = filter_bench(
        df_bench,
        phase=PHASE_MAP["lloyd"],
        language=LANG_CPP,
        samples=max_N,
    )[
        [COL_DIMENSIONS, COL_CLUSTERS, COL_VARIANT, COL_PARAMS, COL_TIME_PER_ALGORITHM_ITER]
    ].copy()

    df_fixed_avg = filter_bench(
        df_bench,
        phase=[PHASE_MAP["soa"], PHASE_MAP["pp"]],
        language=LANG_CPP,
        samples=max_N,
    )[
        [COL_DIMENSIONS, COL_CLUSTERS, COL_VARIANT, COL_PARAMS, COL_PHASE, COL_TIME_S]
    ].copy()

    # Pair fixed costs with the matching Lloyd implementation variant:
    df_plot = df_fixed_avg.merge(
        df_math_summary,
        on=[COL_DIMENSIONS, COL_CLUSTERS, COL_VARIANT, COL_PARAMS],
        validate="many_to_one",
    )

    if df_plot.empty:
        print(
            "Skipping fixed-cost vs Lloyd graph: fixed-cost variants did not match "
            "any C++ Lloyd variants."
        )
    else:
        df_plot[COL_EQUIVALENT_ALGORITHM_ITERS] = df_plot[COL_TIME_S] / df_plot[COL_TIME_PER_ALGORITHM_ITER]
        df_plot[COL_PHASE] = df_plot[COL_PHASE].cat.remove_unused_categories()
        df_plot[COL_VARIANT] = df_plot[COL_VARIANT].cat.remove_unused_categories()

        COL_EQUIVALENT_ALGORITHM_ITERS_INDEX = COL_EQUIVALENT_ALGORITHM_ITERS + "_index"
        COL_TIME_PER_ALGORITHM_ITER_INDEX = COL_TIME_PER_ALGORITHM_ITER + "_index"

        df_plot_norm = df_plot.copy()
        # Baseline Lloyd time/iter per K and variant at the lowest D
        df_plot_norm = df_plot_norm.sort_values([COL_VARIANT, COL_PARAMS, COL_DIMENSIONS, COL_CLUSTERS, COL_PHASE])

        COL_LLOYD_BASELINE = COL_TIME_PER_ALGORITHM_ITER + "_baseline_lowest_dim"

        df_plot_norm[COL_LLOYD_BASELINE] = (
            df_plot_norm
            .groupby([COL_CLUSTERS, COL_VARIANT, COL_PARAMS], observed=True)[COL_TIME_PER_ALGORITHM_ITER]
            .transform("first")
        )

        # Fixed costs expressed in units of the lowest-D Lloyd algorithm iteration
        df_plot_norm[COL_EQUIVALENT_ALGORITHM_ITERS_INDEX] = (
            df_plot_norm[COL_TIME_S] / df_plot_norm[COL_LLOYD_BASELINE]
        )

        # Lloyd algorithm iteration growth, still starting at 1
        df_plot_norm[COL_TIME_PER_ALGORITHM_ITER_INDEX] = (
            df_plot_norm[COL_TIME_PER_ALGORITHM_ITER] / df_plot_norm[COL_LLOYD_BASELINE]
        )

        for (variant, params), df_variant in df_plot_norm.groupby([COL_VARIANT, COL_PARAMS], observed=True):
            if df_variant.empty:
                continue

            plot_keys = (
                df_variant[[COL_CLUSTERS]]
                .drop_duplicates()
                .sort_values([COL_CLUSTERS])
                .reset_index(drop=True)
            )
            fig, axes = create_subplot_grid(
                len(plot_keys),
                cols=2,
                row_height=5,
                fig_width=12,
            )

            for i, row in plot_keys.iterrows():
                k = row[COL_CLUSTERS]
                ax = axes.flatten()[i]
                data_k = filter_bench(df_variant, clusters=k)

                sns.barplot(
                    data=data_k,
                    x=COL_DIMENSIONS,
                    y=COL_EQUIVALENT_ALGORITHM_ITERS_INDEX,
                    hue=COL_PHASE,
                    ax=ax,
                )

                data_line = data_k.drop_duplicates(subset=[COL_DIMENSIONS]).copy()

                sns.pointplot(
                    data=data_line,
                    x=COL_DIMENSIONS,
                    y=COL_TIME_PER_ALGORITHM_ITER_INDEX,
                    ax=ax,
                    color="black",
                    markersize=6,
                    errorbar=None,
                )

                ax.set_title(f"K = {k}", bbox=SMALL_MULTIPLE_TITLE_STYLE)

                if i % 2 == 0:
                    ax.set_ylabel("Cost in baseline Lloyd algorithm iterations")
                else:
                    ax.set_ylabel("")

                ax.set_xlabel("D")
                ax.set_yscale("log")
                # Major ticks/gridlines at 1, 5, 10, 50, 100, ...
                ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 5)))

                # Optional minor ticks/gridlines at the other log positions
                ax.yaxis.set_minor_locator(LogLocator(base=10, subs=(2, 3, 4, 6, 7, 8, 9)))
                ax.yaxis.set_minor_formatter(NullFormatter())

                # Show plain numbers instead of 10^x
                ax.yaxis.set_major_formatter(
                    FuncFormatter(lambda y, _: f"{y:g}")
                )

                ax.grid(axis="y", which="major", linestyle="--", alpha=0.6)
                ax.grid(axis="y", which="minor", linestyle=":", alpha=0.25)

                if i == 0:
                    handles, labels = ax.get_legend_handles_labels()

                    line_handle = Line2D(
                        [0],
                        [0],
                        color="black",
                        marker="o",
                        linestyle="-",
                        label="Lloyd time / iter growth",
                    )

                    ax.legend(
                        handles=handles + [line_handle],
                        labels=labels + ["Lloyd time / iter growth"],
                        title=None,
                    )
                elif ax.get_legend() is not None:
                    ax.get_legend().remove()

            fig.suptitle(
                "Scaling of Fixed Costs vs Lloyd Iteration Time\n"
                f"{format_variant_params_title(variant, params)}; normalized to the lowest D at N = {format_abbrev(max_N)}",
                fontsize=16,
                y=1,
            )

            plt.tight_layout()
            plt.show()


In [ ]:
for phase, df_phase in iter_speedup_phase_data(df_speedup):
    for (variant, params), df_plot in df_phase.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_plot.empty:
            continue

        df_plot = df_plot.copy()
        K_order = sorted(df_plot[COL_CLUSTERS].dropna().unique())

        K_palette = dict(
            zip(
                K_order,
                sns.color_palette("Set1", n_colors=len(K_order)),
            )
        )

        g = sns.relplot(
            data=df_plot,
            x=COL_SAMPLES,
            y=COL_SPEEDUP,
            hue=COL_CLUSTERS,
            hue_order=K_order,
            col=COL_DIMENSIONS,
            col_wrap=3,
            kind="line",
            marker="o",
            palette=K_palette,
            height=4,
            aspect=1.2,
            facet_kws={"sharey": False},
        )

        for dim, ax in g.axes_dict.items():
            subset_dim = df_plot[df_plot[COL_DIMENSIONS] == dim]

            for k, subset in subset_dim.groupby(COL_CLUSTERS, observed=True):
                subset = subset.sort_values(COL_SAMPLES)

                x = subset[COL_SAMPLES].to_numpy()
                y_low = subset[COL_SPEEDUP_CI_LOW].to_numpy()
                y_high = subset[COL_SPEEDUP_CI_HIGH].to_numpy()

                ax.fill_between(
                    x,
                    y_low,
                    y_high,
                    color=K_palette[k],
                    alpha=0.18,
                    linewidth=0,
                )

        style_facet_grid(
            g,
            title=(
                "Median Speedup vs. N with Clustered-Bootstrap CI\n"
                f"Phase: {phase}; {format_variant_params_title(variant, params)}"
            ),
            title_y=1.02,
            x_log=True,
            sample_x_axis=True,
            titles_inside=True,
            grid_axis="both",
        )

        for ax in g.axes.flat:
            ymin, ymax = ax.get_ylim()
            ax.set_ylim(bottom=0, top=max(ymax, 1.05))

            ax.axhline(
                y=1,
                color="black",
                linewidth=3,
                linestyle="-",
                alpha=0.85,
                zorder=10,
            )

        g.set_axis_labels("N", "Speedup Multiplier (x)")

        move_facet_legend_to_row_ends(
            g,
            default_title="K",
            anchor=(1, 0.5),
        )

        plt.show()


In [ ]:
import matplotlib.patches as patches
from numpy.typing import NDArray

def add_speedup_ci_hatching(*, ax, subset, heat, heat_raw):
    ci_width_pivot, _ = make_cluster_pivot(
        subset,
        COL_SPEEDUP_ERROR_WIDTH,
        reference_pivot=heat_raw,
    )

    relative_ci_pivot = ci_width_pivot / heat.abs()
    hatch_fraction_df = relative_ci_pivot.clip(lower=0, upper=1)

    hatch_fraction: NDArray[np.float64] = hatch_fraction_df.to_numpy(
        dtype=np.float64,
        na_value=np.nan,
    )

    for row_idx in range(hatch_fraction.shape[0]):
        for col_idx in range(hatch_fraction.shape[1]):
            frac = hatch_fraction[row_idx, col_idx]

            if np.isfinite(frac) and frac > 0:
                ax.add_patch(
                    patches.Rectangle(
                        (col_idx, row_idx),
                        1,
                        float(frac),
                        facecolor=(1, 1, 1, 0.08),
                        edgecolor=(0, 0, 0, 0.55),
                        hatch="////",
                        linewidth=0,
                        zorder=2,
                    )
                )

    for text in ax.texts:
        text.set_zorder(3)


for phase, df_phase in iter_speedup_phase_data(df_speedup):
    for (variant, params), df_variant in df_phase.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_variant.empty:
            continue

        vmax_val = df_variant[COL_SPEEDUP].max()
        K_values = sorted(df_variant[COL_CLUSTERS].unique())

        legend_handles = [
            patches.Patch(
                facecolor=(1, 1, 1, 0.08),
                edgecolor=(0, 0, 0, 0.55),
                hatch="////",
                label="Hatched area = relative CI width",
            )
        ]

        plot_clustered_heatmap_grid(
            df_variant,
            clusters=K_values,
            value_col=COL_SPEEDUP,
            title=(
                f"Median Speedup Heatmap ({LANG_CPP} vs. {LANG_PY}) by K\n"
                f"Phase: {phase} | Variant: {variant} | Params: {params}"
            ),
            heatmap_kwargs={
                "norm": mcolors.LogNorm(vmin=1, vmax=vmax_val),
                "vmin": 1,
                "vmax": vmax_val,
            },
            cbar_kws={
                "label": "Median Speedup Multiplier (x)",
                "format": mtick.ScalarFormatter(),
                "ticks": [1, 2, 5, 10, 20, 50, 100],
            },
            fmt=".1f",
            legend_handles=legend_handles,
            post_heatmap=add_speedup_ci_hatching,
        )



In [ ]:
def plot_parity_pressure(df, title, legend_handles=None, annot_col=None):
    if df.empty:
        print(f"Skipping {title}: no data.")
        return

    for (variant, params), df_variant in df.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_variant.empty:
            continue

        K_values = sorted(df_variant[COL_CLUSTERS].unique())
        values = df_variant["Parity Pressure"].replace(0, np.nan)

        vmin = min(values.min(), 1.0)
        vmax = 1
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)

        ticks = np.logspace(
            np.floor(np.log10(vmin)),
            np.ceil(np.log10(vmax)),
            int(np.ceil(np.log10(vmax)) - np.floor(np.log10(vmin))) + 1,
        )

        kwargs = {}
        if annot_col is not None:
            kwargs["annot_col"] = annot_col
            kwargs["fmt"] = "" 

        plot_clustered_heatmap_grid(
            df_variant,
            clusters=K_values,
            value_col="Parity Pressure",
            title=(
                title
                + f"\nVariant: {variant}; Params: {params}\n"
                + "values < 1 pass; values > 1 fail or approach failure"
            ),
            heatmap_kwargs={
                "norm": norm,
            },
            cbar_kws={
                "label": "max(diff / tolerance), log scale",
                "ticks": ticks,
                "format": mtick.FormatStrFormatter("%g"),
            },
            legend_handles=legend_handles,
            **kwargs,
        )


if not df_lloyd_parity.empty:
    df_lloyd_pressure = add_lloyd_parity_pressure(df_lloyd_parity)

    plot_parity_pressure(
        df_lloyd_pressure,
        f"Lloyd Parity Pressure ({LANG_CPP} vs. {LANG_PY})",
    )

if not df_gmm_parity.empty:
    df_gmm_pressure = add_gmm_parity_pressure(df_gmm_parity)
    df_gmm_pressure, pressure_letter_handles, pressure_letter_map = (
        add_initial_letter_annotations(
            df_gmm_pressure,
            source_col="Worst Check",
            annot_col="Worst Check Letter",
        )
    )

    plot_parity_pressure(
        df_gmm_pressure,
        f"GMM Parity Pressure ({LANG_CPP} vs. {LANG_PY})\n"
        "max(lower bound, weights, means, covariances, algorithm iterations VS tolerance); ",
        legend_handles=pressure_letter_handles,
        annot_col="Worst Check Letter",
    )

In [ ]:
df_norm = add_speedup_retention(df_speedup)

base_K = df_norm[COL_CLUSTERS].min()
df_norm[COL_PHASE] = df_norm[COL_PHASE].cat.remove_unused_categories()

for phase, df_phase in iter_speedup_phase_data(df_norm):
    for (variant, params), df_variant in df_phase.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_variant.empty:
            continue

        g = sns.relplot(
            data=df_variant,
            x=COL_CLUSTERS,
            y=COL_RETENTION,
            hue=COL_SAMPLES,
            col=COL_DIMENSIONS,
            col_wrap=3,
            kind="line",
            marker="o",
            hue_norm=mcolors.LogNorm(),
            legend="full",
            palette="turbo",
            height=4,
            aspect=1.2,
            facet_kws={"sharey": True},
        )
        
        style_facet_grid(
            g,
            title=(
                f"PHASE: {phase.upper()}\n"
                f"{format_variant_params_title(variant, params)}\n"
                f"Speedup Retention against {base_K}K\n"
                "Does C++ maintain its speed advantage over Python as K increases?"
            ),
            title_y=1.1,
            titles_inside=True,
            grid_axis="y",
            integer_x_axis=True,
        )

        g.set_axis_labels("K", "Retention (%)")
        g.set(ylim=(0, None))

        move_facet_legend_to_row_ends(
            g,
            default_title="N",
            label_formatter=format_abbrev,
            anchor=(1.05, 0.5),
        )

        for ax in g.axes.flat:
            ax.axhline(100, color="red", linestyle="--", alpha=0.5)

        plt.show()


In [ ]:
df_bench = add_time_per_algorithm_iteration_per_sample_columns(
    df_bench,
    statistic="median",
    spread="iqr",
)

median_K = min(
    df_bench[COL_CLUSTERS].unique(),
    key=lambda x: abs(x - df_bench[COL_CLUSTERS].median()),
)

for selected_phase in available_speedup_phases(df_speedup):
    phase_df = filter_bench(
        df_bench,
        clusters=median_K,
        language=LANG_CPP,
        phase=selected_phase,
    ).copy()

    if phase_df.empty:
        continue

    for (variant, params), plot_df in phase_df.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if plot_df.empty:
            continue

        plot_df = plot_df.copy()
        plot_df[COL_PHASE] = plot_df[COL_PHASE].cat.remove_unused_categories()
        plot_df[COL_VARIANT] = plot_df[COL_VARIANT].cat.remove_unused_categories()
        plot_df[COL_PARAMS] = plot_df[COL_PARAMS].cat.remove_unused_categories()
        plot_df = plot_df.sort_values([COL_DIMENSIONS, COL_SAMPLES])

        def draw_spread_band(data, **kwargs):
            data = data.sort_values(COL_SAMPLES)

            ax = plt.gca()
            color = kwargs.get("color", sns.color_palette()[0])

            x = data[COL_SAMPLES].to_numpy()
            y_low = data[COL_TIME_PER_ALGORITHM_ITER_PER_SAMPLE_LOW_MS].to_numpy()
            y_high = data[COL_TIME_PER_ALGORITHM_ITER_PER_SAMPLE_HIGH_MS].to_numpy()

            sns.lineplot(
                data=data,
                x=COL_SAMPLES,
                y=COL_TIME_PER_ALGORITHM_ITER_PER_SAMPLE_MS,
                marker="o",
                errorbar=None,
                ax=ax,
                color=color,
            )

            ax.fill_between(
                x,
                y_low,
                y_high,
                color=color,
                alpha=0.2,
                linewidth=0,
            )

        g = sns.FacetGrid(
            data=plot_df,
            col=COL_DIMENSIONS,
            col_wrap=3,
            sharey=False,
            height=4,
            aspect=1.2,
        )

        g.map_dataframe(draw_spread_band)

        spread_label = plot_df[COL_TIMING_RUN_SPREAD].iloc[0]

        style_facet_grid(
            g,
            title=(
                f"{LANG_CPP} median time per {selected_phase} algorithm iteration per sample "
                "vs. N\n"
                f"Variant: {variant}; Params: {params}; error bars show timing-run spread ({spread_label}); "
                f"K = {median_K}"
            ),
            title_y=1.15,
            x_log=True,
            sample_x_axis=True,
        )

        g.set_axis_labels(
            "N",
            f"Time per {selected_phase} algorithm iteration per sample (ms)",
        )

        plt.show()

In [ ]:
import matplotlib.ticker as mtick

from python.benchmark_reporting.lloyd_modeling import (
    COL_MODEL_NAME,
    COL_COEFFICIENT,
    COL_PRED_TIME_PER_ALGORITHM_ITER_MS,
    COL_WORK_SIZE,
    fit_lloyd_timing_models,
)

lloyd_model_report = fit_lloyd_timing_models(
    df_lloyd,
    vif_threshold=100.0,
    coefficient_threshold=1e-5,
)

df_model_predictions = lloyd_model_report.predictions
df_model_summary = lloyd_model_report.summary
df_model_coefficients_all = lloyd_model_report.coefficients_all
df_model_coefficients = lloyd_model_report.coefficients
df_lloyd_collinearity_removed = lloyd_model_report.collinearity_removed

print(df_model_summary[COL_MODEL_NAME][0])
display(df_model_summary.drop(columns=[COL_MODEL_NAME], errors="ignore"))
display(df_model_coefficients.drop(columns=[COL_MODEL_NAME], errors="ignore"))


# --- Visualization ----------------------------------------------------------

for (variant, params), df_plot in df_model_predictions.groupby([COL_VARIANT, COL_PARAMS], observed=True):
    if df_plot.empty:
        continue

    df_cpp = filter_bench(df_plot, language=LANG_CPP).copy()
    df_py = filter_bench(df_plot, language=LANG_PY).copy()

    fig, ax = plt.subplots(figsize=(12, 7))

    sns.scatterplot(
        data=df_cpp,
        x=COL_WORK_SIZE,
        y=COL_PRED_TIME_PER_ALGORITHM_ITER_MS,
        marker="X",
        s=50,
        color="black",
        alpha=0.3,
        label=f"{LANG_CPP} fitted model",
        ax=ax,
    )

    sns.scatterplot(
        data=df_py,
        x=COL_WORK_SIZE,
        y=COL_PRED_TIME_PER_ALGORITHM_ITER_MS,
        marker="P",
        s=50,
        color="black",
        alpha=0.3,
        label=f"{LANG_PY} fitted model",
        ax=ax,
    )

    sns.scatterplot(
        data=df_plot,
        x=COL_WORK_SIZE,
        y=COL_TIME_PER_ALGORITHM_ITER_MS,
        hue=COL_LANGUAGE,
        alpha=1,
        s=25,
        ax=ax,
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.xaxis.set_major_formatter(
        mtick.FuncFormatter(lambda x, _: format_abbrev(x))
    )

    ax.set_xlabel("Estimated Lloyd dataset size = D × N")
    ax.set_ylabel("Time per Lloyd algorithm iteration (ms)")
    ax.set_title(
        "Lloyd algorithm iteration timing models by language\n"
        f"{format_variant_params_title(variant, params)}"
    )

    ax.grid(axis="both", linestyle="--", alpha=0.7)
    ax.legend()

    plt.tight_layout()
    plt.show()
